# AXIOM Constitutional AI — Qwen2.5-Coder Fine-Tune

Fine-tunes Qwen2.5-Coder-1.5B-Instruct on AXIOM constitutional AI examples using QLoRA + ChatML.

**Pipeline:** Upload data → ChatML conversion → Load model (4-bit) → LoRA adapters → Train → Merge → GGUF → Download

**Data formats accepted:**
- `train_qwen_chatml.jsonl` — pre-formatted `{"messages": [{"role":..., "content":...}]}`
- `axiom_training_data.jsonl` — raw `{"instruction", "input", "output"}` → auto-converted to ChatML
- `autotrain_data/train.jsonl` — TinyLlama ChatML `{"text": "<|im_start|>..."}` → parsed to messages

**Requirements:** Colab T4 GPU (free tier), ~10-15 minutes training time

**Source:** github.com/Orivael-Dev/axiom | Patent Pending ORVL-001-PROV

In [ ]:
# Cell 1: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to: Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Cell 2: Install dependencies
# Pin versions — these are tested and working on Colab T4
!pip install -q transformers==4.46.0 peft==0.13.0 trl==0.12.0
!pip install -q bitsandbytes accelerate==1.0.0
!pip install -q datasets scipy sentencepiece protobuf

import bitsandbytes as bnb
import torch
print(f"bitsandbytes: {bnb.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"bnb CUDA setup: OK")
# Pull the AXIOM Colab adapter so Cell 3 can call load_training_data().
# Source: notebooks/axiom_colab.py + notebooks/sample_training_data.jsonl
!curl -fsSL -o axiom_colab.py https://raw.githubusercontent.com/Orivael-Dev/axiom/main/notebooks/axiom_colab.py
!curl -fsSL -o sample_training_data.jsonl https://raw.githubusercontent.com/Orivael-Dev/axiom/main/notebooks/sample_training_data.jsonl


In [ ]:
# Cell 3: Load training data via the AXIOM Colab adapter.
#
# load_training_data() auto-picks a source for the current
# environment:
#   - Drive at /content/drive/MyDrive/axiom_training_data.jsonl if mounted
#   - Colab file picker otherwise
#   - bundled sample as a last resort (so the rest of the notebook
#     is runnable end-to-end with zero setup)
#
# Explicit sources:
#   load_training_data('upload')                        # force picker
#   load_training_data('drive:/some_folder/train.jsonl') # Drive path
#   load_training_data('hf:tatsu-lab/alpaca#train[:500]')# HuggingFace hub
#   load_training_data('https://.../data.jsonl')         # raw URL
#   load_training_data('sample')                         # bundled 50 ex
#   load_training_data('/content/local.jsonl')           # local path
#
# Input shapes auto-detected: {messages}, {instruction, input, output}
# (raw alpaca-style), or {text: '<|im_start|>...'} (ChatML strings).
from axiom_colab import load_training_data

examples = load_training_data()

print(f"\n{len(examples)} ChatML examples ready")
roles = [m['role'] for m in examples[0]['messages']]
print(f"First-row roles: {roles}")
print(f"User preview: {examples[0]['messages'][1]['content'][:200]}...")


In [ ]:
# Cell 4: Prepare HuggingFace Dataset — 90/10 train/eval split
from datasets import Dataset

dataset = Dataset.from_list(examples)
dataset = dataset.shuffle(seed=42)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} examples")
print(f"Eval:  {len(eval_dataset)} examples")

# Token length estimate from messages content
lengths = [sum(len(m['content']) for m in ex['messages']) for ex in examples]
print(f"\nChar lengths — min: {min(lengths)}, max: {max(lengths)}, median: {sorted(lengths)[len(lengths)//2]}")
print(f"Est tokens — total: {sum(lengths)//4:,}, avg: {sum(lengths)//len(lengths)//4}")

In [ ]:
# Cell 5: Load Qwen2.5-Coder in 4-bit (QLoRA)
# 1.5B fits T4 easily (~1.2 GB VRAM in 4-bit)
# Change to "Qwen/Qwen2.5-Coder-7B-Instruct" if you have A100/L4
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Chat template: {'yes' if tokenizer.chat_template else 'no'}")

In [ ]:
# Cell 6: Configure LoRA adapters
# Qwen2.5 uses standard transformer attention
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 7: Apply Qwen's chat template to format messages -> text
# Qwen2.5 uses ChatML: <|im_start|>role\ncontent<|im_end|>
# tokenizer.apply_chat_template handles this natively

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_formatted = train_dataset.map(format_chat)
eval_formatted  = eval_dataset.map(format_chat)

print(f"Formatted {len(train_formatted)} train + {len(eval_formatted)} eval")
print(f"\nChatML sample:")
print(train_formatted[0]['text'][:500])
print("...")

In [ ]:
# Cell 8: Training config
# 4 epochs, effective batch 8, cosine LR 2e-4, eval every 20 steps
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./axiom-qwen-lora",

    # Duration
    num_train_epochs=4,

    # Batch (effective = 1 * 8 = 8)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Sequence
    max_seq_length=2048,

    # Precision (T4 supports fp16, not bf16)
    fp16=True,
    bf16=False,

    # Evaluation
    eval_strategy="steps",
    eval_steps=20,

    # Checkpoints
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=5,
    report_to="none",

    # Optimizer
    optim="paged_adamw_8bit",
    weight_decay=0.05,
    max_grad_norm=0.3,

    # Dataset
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    tokenizer=tokenizer,
)

print(f"Ready to train.")

In [ ]:
# Cell 9: Train
import time

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final train loss: {trainer.state.log_history[-1].get('train_loss', 'N/A')}")
eval_losses = [h.get('eval_loss', 999) for h in trainer.state.log_history if 'eval_loss' in h]
if eval_losses:
    print(f"Best eval loss:   {min(eval_losses):.4f}")

In [ ]:
# Cell 10: Save LoRA adapters
import os

trainer.save_model("./axiom-qwen-lora/final")
tokenizer.save_pretrained("./axiom-qwen-lora/final")
print("LoRA adapters saved to ./axiom-qwen-lora/final")

adapter_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk("./axiom-qwen-lora/final")
    for f in fns
) / (1024 * 1024)
print(f"Adapter size: {adapter_size:.1f} MB")

In [ ]:
# Cell 11: Merge LoRA into base model
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, os

# Reload base in float16 for clean merge
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Merge LoRA weights into base
model = PeftModel.from_pretrained(base_model, "./axiom-qwen-lora/final")
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("./axiom-qwen-merged")
tok = AutoTokenizer.from_pretrained("./axiom-qwen-lora/final", trust_remote_code=True)
tok.save_pretrained("./axiom-qwen-merged")

merged_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk("./axiom-qwen-merged")
    for f in fns
) / (1024**3)
print(f"Merged model saved: {merged_size:.2f} GB")

In [ ]:
# Cell 12: Convert to GGUF — F16, Q8_0, and Q4_K_M
# Produces all three so you can pick the right tradeoff:
#   F16     ~3.0 GB  lossless, best quality, needs more RAM
#   Q8_0    ~1.6 GB  near-lossless, good balance
#   Q4_K_M  ~1.0 GB  smallest, fast inference, slight quality loss
import subprocess, os, shutil

# Clean up any partial clone from previous attempts
if os.path.exists("llama.cpp") and not os.path.exists("llama.cpp/convert_hf_to_gguf.py"):
    print("Removing incomplete llama.cpp clone...")
    shutil.rmtree("llama.cpp")

# Install GGUF conversion tools
!pip install -q gguf sentencepiece

# Clone llama.cpp (retry-safe)
if not os.path.exists("llama.cpp"):
    print("Cloning llama.cpp...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git"],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed: {result.stderr}")

assert os.path.exists("llama.cpp/convert_hf_to_gguf.py"), \
    "convert_hf_to_gguf.py not found — clone may have failed"
print("llama.cpp ready")

# Step 1: HF -> GGUF F16 (base for all quantizations)
print("\n[1/4] Converting to GGUF F16...")
!python llama.cpp/convert_hf_to_gguf.py \
    ./axiom-qwen-merged \
    --outfile axiom-qwen-f16.gguf \
    --outtype f16

assert os.path.exists("axiom-qwen-f16.gguf"), \
    "F16 conversion failed — check output above"
f16_mb = os.path.getsize("axiom-qwen-f16.gguf") / (1024 * 1024)
print(f"F16 GGUF: {f16_mb:.0f} MB")

# Step 2: Build quantize tool
print("\n[2/4] Building llama-quantize...")
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -3 && \
    cmake --build build --target llama-quantize -j$(nproc) 2>&1 | tail -5

QUANTIZE_BIN = "llama.cpp/build/bin/llama-quantize"
assert os.path.exists(QUANTIZE_BIN), \
    "llama-quantize build failed — check cmake output above"
print("llama-quantize built")

# Step 3: Quantize to Q8_0 (8-bit)
print("\n[3/4] Quantizing to Q8_0 (8-bit)...")
!./llama.cpp/build/bin/llama-quantize \
    axiom-qwen-f16.gguf \
    axiom-qwen-Q8_0.gguf \
    Q8_0

assert os.path.exists("axiom-qwen-Q8_0.gguf"), \
    "Q8_0 quantization failed — check output above"
q8_mb = os.path.getsize("axiom-qwen-Q8_0.gguf") / (1024 * 1024)
print(f"Q8_0 GGUF: {q8_mb:.0f} MB")

# Step 4: Quantize to Q4_K_M (4-bit)
print("\n[4/4] Quantizing to Q4_K_M (4-bit)...")
!./llama.cpp/build/bin/llama-quantize \
    axiom-qwen-f16.gguf \
    axiom-qwen-Q4_K_M.gguf \
    Q4_K_M

assert os.path.exists("axiom-qwen-Q4_K_M.gguf"), \
    "Q4_K_M quantization failed — check output above"
q4_mb = os.path.getsize("axiom-qwen-Q4_K_M.gguf") / (1024 * 1024)
print(f"Q4_K_M GGUF: {q4_mb:.0f} MB")

# Summary
print(f"\n{'='*50}")
print(f"  GGUF Output Summary")
print(f"{'='*50}")
print(f"  F16     axiom-qwen-f16.gguf      {f16_mb:>7.0f} MB  (lossless)")
print(f"  Q8_0    axiom-qwen-Q8_0.gguf     {q8_mb:>7.0f} MB  (8-bit)")
print(f"  Q4_K_M  axiom-qwen-Q4_K_M.gguf   {q4_mb:>7.0f} MB  (4-bit)")
print(f"{'='*50}")

In [ ]:
# Cell 13: Download GGUF — pick your quantization
# Change QUANT to download a different version:
#   "Q4_K_M"  ~1.0 GB  fastest, smallest         (default)
#   "Q8_0"    ~1.6 GB  near-lossless, recommended
#   "F16"     ~3.0 GB  full precision, best quality
from google.colab import files

QUANT = "Q4_K_M"  # <-- change to "Q8_0" or "F16"

quant_map = {
    "Q4_K_M": "axiom-qwen-Q4_K_M.gguf",
    "Q8_0":   "axiom-qwen-Q8_0.gguf",
    "F16":    "axiom-qwen-f16.gguf",
}

gguf_file = quant_map[QUANT]
size_mb = os.path.getsize(gguf_file) / (1024 * 1024)
print(f"Downloading {gguf_file} ({size_mb:.0f} MB)...")
files.download(gguf_file)

print(f"\nOllama deployment:")
print(f"  ollama create axiom-qwen -f Modelfile.qwen")
print(f"  ollama run axiom-qwen")
print(f"\nModelfile.qwen should reference: {gguf_file}")